### Numerical integration of the non-passage density

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ssms.basic_simulators.race_math import big_F, q, small_f

mu = 0.75
sigma = 1.0
x0 = 0.0
a = 1.0
b = 0.0
T = 1.5

# Finite replacements for the mathematical lower bound -infinity.
# For this example, -100 is the conservative default; the plots below check smaller cutoffs.
lower_bounds = np.array([-4.0, -10.0, -25.0, -50.0, -100.0])
n_trapezoid_points = 20_000
n_gauss_nodes = 128

In [ ]:
gauss_nodes, gauss_weights = np.polynomial.legendre.leggauss(n_gauss_nodes)

def integrate_trapezoid(t, lower_x):
    x_grid = np.linspace(lower_x, a + b * t, n_trapezoid_points)
    return np.trapezoid(q(x_grid, mu, sigma, a, b, t, x0), x_grid)

def integrate_gauss_legendre(t, lower_x):
    upper_x = a + b * t
    x_grid = 0.5 * (upper_x - lower_x) * gauss_nodes + 0.5 * (upper_x + lower_x)
    weights = 0.5 * (upper_x - lower_x) * gauss_weights
    return np.sum(weights * q(x_grid, mu, sigma, a, b, t, x0))

ts = np.linspace(1e-3, T, 300)
survival = 1.0 - big_F(ts, mu, sigma, a, b, T, x0)
trapezoid_mass = np.array([[integrate_trapezoid(t, lower_x) for t in ts] for lower_x in lower_bounds])
gauss_mass = np.array([[integrate_gauss_legendre(t, lower_x) for t in ts] for lower_x in lower_bounds])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for i, lower_x in enumerate(lower_bounds):
    ax[0].plot(ts, trapezoid_mass[i], label=fr'$L={lower_x:g}$')
    ax[1].plot(ts, gauss_mass[i], label=fr'$L={lower_x:g}$')
for axis, title in zip(ax, ['trapezoid', 'Gauss-Legendre']):
    axis.plot(ts, survival, color='black', linestyle='--', linewidth=2, label=r'$1-F(t)$')
    axis.set(title=title, xlabel=r'$t$', ylim=(0, 1))
    axis.legend(fontsize=8)
ax[0].set_ylabel('non-passage probability')
plt.show()

In [ ]:
def integrate_f_trapezoid(t):
    t_grid = np.linspace(1e-8, t, n_trapezoid_points)
    return np.trapezoid(small_f(t_grid, mu, sigma, a, b, T, x0), t_grid)

def integrate_f_gauss_legendre(t):
    t_grid = 0.5 * t * (gauss_nodes + 1.0)
    weights = 0.5 * t * gauss_weights
    return np.sum(weights * small_f(t_grid, mu, sigma, a, b, T, x0))

f_cdf_trapezoid = np.array([integrate_f_trapezoid(t) for t in ts])
f_cdf_gauss = np.array([integrate_f_gauss_legendre(t) for t in ts])
analytic_cdf = big_F(ts, mu, sigma, a, b, T, x0)

print(f'F(T) from trapezoid:     {f_cdf_trapezoid[-1]:.6f}')
print(f'F(T) from Gauss-Legendre: {f_cdf_gauss[-1]:.6f}')
print(f'analytic F(T):            {analytic_cdf[-1]:.6f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ts, f_cdf_trapezoid, color='tab:orange', label=r'$\int_0^t f(s)\,ds$: trapezoid')
ax.plot(ts, f_cdf_gauss, color='tab:green', label=r'$\int_0^t f(s)\,ds$: Gauss-Legendre')
ax.plot(ts, analytic_cdf, color='black', linestyle='--', label=r'$F(t)$')
ax.set(xlabel=r'$t$', ylabel='CDF', ylim=(0, 1))
ax.legend(fontsize=8)
plt.show()

In [ ]:
target_T = survival[-1]
trapezoid_error = np.abs(trapezoid_mass[:, -1] - target_T)
gauss_error = np.abs(gauss_mass[:, -1] - target_T)

print('lower bound    trapezoid       Gauss-Legendre')
for lower_x, trap, gauss in zip(lower_bounds, trapezoid_mass[:, -1], gauss_mass[:, -1]):
    print(f'{lower_x:10.0f}    {trap:.8f}       {gauss:.8f}')
print(f'1 - F(T) = {target_T:.8f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(-lower_bounds, trapezoid_error, 'o-', label='trapezoid')
ax.semilogy(-lower_bounds, gauss_error, 's-', label='Gauss-Legendre')
ax.set(xlabel=r'magnitude of lower cutoff $-L$', ylabel=r'absolute error at $T$')
ax.legend()
plt.show()